# ML Fraud Detection with Feast, Flyte, and XGboost

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/fraud-detection-feast/ml-fraud-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Build a fraud detection ML pipeline that:
1. **Downloads & prepares** the Sparkov credit card fraud dataset (~500K transactions)
2. **Trains an XGBoost model** with engineered features and class imbalance handling
3. **Materializes user profiles** to a Feast feature store for real-time lookups
4. **Deploys a scoring API** that combines model predictions with Feast features
5. **Deploys a dashboard** for interactive fraud scoring

All orchestrated with **Flyte**: runs locally for development, deploys to a cluster for production.


> This notebook is made with intention of running in Google colab that connects to a hosted Flyte or Union.ai cluster, for local setup checkout the repo and README
https://github.com/unionai/workshops/blob/main/tutorials/fraud-detection-feast

In [ ]:
# Setup for running in colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    #setup project and keyring if in colab env
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/fraud-detection-feast
    !uv pip install -r requirements.txt
    !uv pip install -r requirements.txt keyrings.alt
    !mkdir -p ~/.config/python_keyring && echo -e "[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring" > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

## Connect to the Flyte Cluster

Create a config that points to the workshop cluster. This uses **headless auth** (no browser needed in Colab). You'll get a link to visit to authenticate.

The config tells Flyte:
- **endpoint**: which cluster to connect to
- **project / domain**: where to run your tasks
- **builder: remote**: build container images on the cluster (no local Docker needed)

In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --project workshopfeast \
    --domain development \
    --builder remote \
    --auth-type headless

## Run the Pipeline

This single command runs the full fraud detection pipeline on the cluster:

```
fraud_detection_pipeline
  ├── prepare_data           > download dataset, engineer features
  ├── train_model            > XGBoost classifier > model.joblib
  └── materialize_features   > Feast apply + materialize > feast_artifacts/
```

Steps 2 and 3 run **in parallel** since they both only depend on step 1.

The first run builds a container image with all dependencies, which takes a few minutes. Subsequent runs reuse the cached image.

**Check the Flyte UI** (link in the output) to see live reports: stat cards, confusion matrix, feature importance chart, and a pipeline step indicator.

In [ ]:
!flyte run workflow.py fraud_detection_pipeline --n_estimators 500 --max_depth 8 

## Deploy the Scoring App

Deploy a FastAPI app that scores transactions in real time. The app uses `RunOutput` to automatically pull the trained model and Feast artifacts from the pipeline run above, so there's no manual file copying.

**How scoring works:**
- **From the request**: transaction amount, merchant category, location, time
- **From Feast**: user's spending history, home location, age (pre-materialized)
- **Computed at scoring time**: amount z-score, distance from home
- **Rule overrides**: business rules catch extreme cases the model can't extrapolate to

Once deployed, test it with curl:
```bash
# Normal transaction
curl "<app-url>/score?user_id=42&amt=25&category=grocery_pos&merch_lat=33.9&merch_long=-80.3"

# Suspicious: large amount + far away + late night
curl "<app-url>/score?user_id=42&amt=9999&category=shopping_net&merch_lat=48.8&merch_long=2.3&hour=23"
```

In [ ]:
!flyte deploy app.py serving_env


#!RUN_NAME=r94jl7hxd7vxk6gdprwx flyte deploy app.py serving_env 

## Deploy the Dashboard

Deploy a Gradio dashboard that provides an interactive UI for the scoring API. It uses `AppEndpoint(app_name="fraud-scorer")` to auto-discover the scoring app URL, so there are no hardcoded endpoints.

Adjust user ID, amount, category, and merchant location in the UI to see fraud predictions, risk signals, and user profiles in real time.

In [ ]:
!flyte deploy dashboard.py dashboard_env

## What Just Happened?

With three commands you went from raw data to a production fraud detection system:

| Step | Command | What it does |
|------|---------|-------------|
| Pipeline | `flyte run workflow.py fraud_detection_pipeline` | Downloads data, trains XGBoost, materializes Feast features |
| Scoring API | `flyte deploy app.py serving_env` | Deploys FastAPI app that combines model + Feast for real-time scoring |
| Dashboard | `flyte deploy dashboard.py dashboard_env` | Deploys interactive Gradio UI that calls the scoring API |

**Key patterns demonstrated:**
- **Feast feature store**: same features for training and serving, no skew
- **RunOutput**: apps automatically pull artifacts from the latest pipeline run
- **AppEndpoint**: apps discover each other by name, no hardcoded URLs
- **Parallel tasks**: model training and feature materialization run concurrently

### Next Steps

- Check the [README](https://github.com/unionai/workshops/blob/main/tutorials/fraud-detection-feast/README.md) for local development setup, model tuning details, and architecture deep-dive
- Try swapping XGBoost for a different model in `workflow.py`
- Add new features to the Feast store and retrain
- Pin to a specific model version: `flyte deploy app.py serving_env -- --run-name <run_name>`